In [1]:
# imports
import pyomo.environ as pe
import pyomo.opt as po
import math
from itertools import permutations

# Inputs

In [2]:
# -------------------------------------------------------------------------
# SETS
# -------------------------------------------------------------------------
factories = ['IJmuiden', 'Segal', 'South Wales']
customer_areas = ['Bochum', 'Boenen', 'Dortmund', 'Gelsenkirchen','Hagen', 'Iserlohn', 'Neuss', 'Schwerte']
long_bar_types = ['1', '2']
rebar_types = ['A', 'B', 'C']
numOfPeriods = 4
vehicles = [1, 2]   # 2 vehicles per factory

# Parameters needed for steel capacity/fleet capacity calculations
length_lb = {'1': 9, '2': 12}
length_rb = {'A': 2.4, 'B': 3.6, 'C': 4.2}
diameter = {'1': 0.057, '2': 0.057, 'A': 0.057, 'B': 0.057, 'C': 0.057}
density = 7.85 # t/m3

# -------------------------------------------------------------------------
# Steel and Vehicle capacity
# -------------------------------------------------------------------------
steel_capacity = {'IJmuiden': 12, 'Segal': 10, 'South Wales': 28}
vehicle_capacity = 10  # tons per vehicle

# -------------------------------------------------------------------------
# Transportation costs
# -------------------------------------------------------------------------
f_fleet = {'IJmuiden': 130, 'Segal': 150, 'South Wales': 100}
c_fleet = {'IJmuiden': 3, 'Segal': 3, 'South Wales': 3}

# -------------------------------------------------------------------------
# Demand
# -------------------------------------------------------------------------
demand = {
    # Rebar A
    ('A', 1, 'Bochum'): 2,  ('A', 1, 'Boenen'): 4,  ('A', 1, 'Dortmund'): 2,  ('A', 1, 'Gelsenkirchen'): 5,
    ('A', 1, 'Hagen'): 19,  ('A', 1, 'Iserlohn'): 13, ('A', 1, 'Neuss'): 20, ('A', 1, 'Schwerte'): 4,

    ('A', 2, 'Bochum'): 6,  ('A', 2, 'Boenen'): 8,  ('A', 2, 'Dortmund'): 7,  ('A', 2, 'Gelsenkirchen'): 5,
    ('A', 2, 'Hagen'): 23,  ('A', 2, 'Iserlohn'): 19, ('A', 2, 'Neuss'): 16, ('A', 2, 'Schwerte'): 5,

    ('A', 3, 'Bochum'): 5,  ('A', 3, 'Boenen'): 5,  ('A', 3, 'Dortmund'): 6,  ('A', 3, 'Gelsenkirchen'): 5,
    ('A', 3, 'Hagen'): 25,  ('A', 3, 'Iserlohn'): 17, ('A', 3, 'Neuss'): 14, ('A', 3, 'Schwerte'): 3,

    ('A', 4, 'Bochum'): 3,  ('A', 4, 'Boenen'): 10, ('A', 4, 'Dortmund'): 5,  ('A', 4, 'Gelsenkirchen'): 5,
    ('A', 4, 'Hagen'): 16,  ('A', 4, 'Iserlohn'): 14, ('A', 4, 'Neuss'): 26, ('A', 4, 'Schwerte'): 4,

    # Rebar B
    ('B', 1, 'Bochum'): 4,  ('B', 1, 'Boenen'): 5,  ('B', 1, 'Dortmund'): 4,  ('B', 1, 'Gelsenkirchen'): 9,
    ('B', 1, 'Hagen'): 15,  ('B', 1, 'Iserlohn'): 22, ('B', 1, 'Neuss'): 12, ('B', 1, 'Schwerte'): 2,

    ('B', 2, 'Bochum'): 5,  ('B', 2, 'Boenen'): 8,  ('B', 2, 'Dortmund'): 5,  ('B', 2, 'Gelsenkirchen'): 10,
    ('B', 2, 'Hagen'): 33,  ('B', 2, 'Iserlohn'): 26, ('B', 2, 'Neuss'): 23, ('B', 2, 'Schwerte'): 8,

    ('B', 3, 'Bochum'): 7,  ('B', 3, 'Boenen'): 12, ('B', 3, 'Dortmund'): 8,  ('B', 3, 'Gelsenkirchen'): 6,
    ('B', 3, 'Hagen'): 31,  ('B', 3, 'Iserlohn'): 20, ('B', 3, 'Neuss'): 30, ('B', 3, 'Schwerte'): 2,

    ('B', 4, 'Bochum'): 8,  ('B', 4, 'Boenen'): 13, ('B', 4, 'Dortmund'): 10, ('B', 4, 'Gelsenkirchen'): 6,
    ('B', 4, 'Hagen'): 33,  ('B', 4, 'Iserlohn'): 27, ('B', 4, 'Neuss'): 30, ('B', 4, 'Schwerte'): 6,

    # Rebar C
    ('C', 1, 'Bochum'): 6,  ('C', 1, 'Boenen'): 6,  ('C', 1, 'Dortmund'): 7,  ('C', 1, 'Gelsenkirchen'): 10,
    ('C', 1, 'Hagen'): 12,  ('C', 1, 'Iserlohn'): 14, ('C', 1, 'Neuss'): 22, ('C', 1, 'Schwerte'): 5,

    ('C', 2, 'Bochum'): 7,  ('C', 2, 'Boenen'): 10, ('C', 2, 'Dortmund'): 6,  ('C', 2, 'Gelsenkirchen'): 9,
    ('C', 2, 'Hagen'): 35,  ('C', 2, 'Iserlohn'): 25, ('C', 2, 'Neuss'): 32, ('C', 2, 'Schwerte'): 6,

    ('C', 3, 'Bochum'): 7,  ('C', 3, 'Boenen'): 15, ('C', 3, 'Dortmund'): 4,  ('C', 3, 'Gelsenkirchen'): 9,
    ('C', 3, 'Hagen'): 33,  ('C', 3, 'Iserlohn'): 23, ('C', 3, 'Neuss'): 31, ('C', 3, 'Schwerte'): 7,

    ('C', 4, 'Bochum'): 7,  ('C', 4, 'Boenen'): 12, ('C', 4, 'Dortmund'): 12, ('C', 4, 'Gelsenkirchen'): 10,
    ('C', 4, 'Hagen'): 38,  ('C', 4, 'Iserlohn'): 24, ('C', 4, 'Neuss'): 31, ('C', 4, 'Schwerte'): 2,
}

# -------------------------------------------------------------------------
# Distance between factories and customer areas in km
# -------------------------------------------------------------------------
dist = {
    # From IJmuiden
    ('IJmuiden', 'IJmuiden'): 0,
    ('IJmuiden', 'Segal'): 284,
    ('IJmuiden', 'South Wales'): 826,
    ('IJmuiden', 'Bochum'): 250,
    ('IJmuiden', 'Boenen'): 282,
    ('IJmuiden', 'Dortmund'): 266,
    ('IJmuiden', 'Gelsenkirchen'): 234,
    ('IJmuiden', 'Hagen'): 289,
    ('IJmuiden', 'Iserlohn'): 299,
    ('IJmuiden', 'Neuss'): 259,
    ('IJmuiden', 'Schwerte'): 279,

    # From Segal
    ('Segal', 'IJmuiden'): 284,
    ('Segal', 'Segal'): 0,
    ('Segal', 'South Wales'): 750,
    ('Segal', 'Bochum'): 203,
    ('Segal', 'Boenen'): 242,
    ('Segal', 'Dortmund'): 222,
    ('Segal', 'Gelsenkirchen'): 198,
    ('Segal', 'Hagen'): 206,
    ('Segal', 'Iserlohn'): 226,
    ('Segal', 'Neuss'): 140,
    ('Segal', 'Schwerte'): 216,

    # From South Wales
    ('South Wales', 'IJmuiden'): 826,
    ('South Wales', 'Segal'): 750,
    ('South Wales', 'South Wales'): 0,
    ('South Wales', 'Bochum'): 866,
    ('South Wales', 'Boenen'): 914,
    ('South Wales', 'Dortmund'): 885,
    ('South Wales', 'Gelsenkirchen'): 859,
    ('South Wales', 'Hagen'): 903,
    ('South Wales', 'Iserlohn'): 913,
    ('South Wales', 'Neuss'): 843,
    ('South Wales', 'Schwerte'): 901,

    # From Bochum
    ('Bochum', 'IJmuiden'): 250,
    ('Bochum', 'Segal'): 203,
    ('Bochum', 'South Wales'): 866,
    ('Bochum', 'Bochum'): 0,
    ('Bochum', 'Boenen'): 55,
    ('Bochum', 'Dortmund'): 21,
    ('Bochum', 'Gelsenkirchen'): 19,
    ('Bochum', 'Hagen'): 41,
    ('Bochum', 'Iserlohn'): 51,
    ('Bochum', 'Neuss'): 56,
    ('Bochum', 'Schwerte'): 39,

    # From Boenen
    ('Boenen', 'IJmuiden'): 282,
    ('Boenen', 'Segal'): 242,
    ('Boenen', 'South Wales'): 914,
    ('Boenen', 'Bochum'): 55,
    ('Boenen', 'Boenen'): 0,
    ('Boenen', 'Dortmund'): 34,
    ('Boenen', 'Gelsenkirchen'): 56,
    ('Boenen', 'Hagen'): 44,
    ('Boenen', 'Iserlohn'): 54,
    ('Boenen', 'Neuss'): 102,
    ('Boenen', 'Schwerte'): 32,

    # From Dortmund
    ('Dortmund', 'IJmuiden'): 266,
    ('Dortmund', 'Segal'): 222,
    ('Dortmund', 'South Wales'): 885,
    ('Dortmund', 'Bochum'): 21,
    ('Dortmund', 'Boenen'): 34,
    ('Dortmund', 'Dortmund'): 0,
    ('Dortmund', 'Gelsenkirchen'): 34,
    ('Dortmund', 'Hagen'): 21,
    ('Dortmund', 'Iserlohn'): 36,
    ('Dortmund', 'Neuss'): 75,
    ('Dortmund', 'Schwerte'): 15,

    # From Gelsenkirchen
    ('Gelsenkirchen', 'IJmuiden'): 234,
    ('Gelsenkirchen', 'Segal'): 198,
    ('Gelsenkirchen', 'South Wales'): 859,
    ('Gelsenkirchen', 'Bochum'): 19,
    ('Gelsenkirchen', 'Boenen'): 56,
    ('Gelsenkirchen', 'Dortmund'): 34,
    ('Gelsenkirchen', 'Gelsenkirchen'): 0,
    ('Gelsenkirchen', 'Hagen'): 56,
    ('Gelsenkirchen', 'Iserlohn'): 66,
    ('Gelsenkirchen', 'Neuss'): 62,
    ('Gelsenkirchen', 'Schwerte'): 54,

    # From Hagen
    ('Hagen', 'IJmuiden'): 289,
    ('Hagen', 'Segal'): 206,
    ('Hagen', 'South Wales'): 903,
    ('Hagen', 'Bochum'): 41,
    ('Hagen', 'Boenen'): 44,
    ('Hagen', 'Dortmund'): 21,
    ('Hagen', 'Gelsenkirchen'): 56,
    ('Hagen', 'Hagen'): 0,
    ('Hagen', 'Iserlohn'): 20,
    ('Hagen', 'Neuss'): 66,
    ('Hagen', 'Schwerte'): 14,

    # From Iserlohn
    ('Iserlohn', 'IJmuiden'): 299,
    ('Iserlohn', 'Segal'): 226,
    ('Iserlohn', 'South Wales'): 913,
    ('Iserlohn', 'Bochum'): 51,
    ('Iserlohn', 'Boenen'): 54,
    ('Iserlohn', 'Dortmund'): 36,
    ('Iserlohn', 'Gelsenkirchen'): 66,
    ('Iserlohn', 'Hagen'): 20,
    ('Iserlohn', 'Iserlohn'): 0,
    ('Iserlohn', 'Neuss'): 85,
    ('Iserlohn', 'Schwerte'): 15,

    # From Neuss
    ('Neuss', 'IJmuiden'): 259,
    ('Neuss', 'Segal'): 140,
    ('Neuss', 'South Wales'): 843,
    ('Neuss', 'Bochum'): 56,
    ('Neuss', 'Boenen'): 102,
    ('Neuss', 'Dortmund'): 75,
    ('Neuss', 'Gelsenkirchen'): 62,
    ('Neuss', 'Hagen'): 66,
    ('Neuss', 'Iserlohn'): 85,
    ('Neuss', 'Neuss'): 0,
    ('Neuss', 'Schwerte'): 75,

    # From Schwerte
    ('Schwerte', 'IJmuiden'): 279,
    ('Schwerte', 'Segal'): 216,
    ('Schwerte', 'South Wales'): 901,
    ('Schwerte', 'Bochum'): 39,
    ('Schwerte', 'Boenen'): 32,
    ('Schwerte', 'Dortmund'): 15,
    ('Schwerte', 'Gelsenkirchen'): 54,
    ('Schwerte', 'Hagen'): 14,
    ('Schwerte', 'Iserlohn'): 15,
    ('Schwerte', 'Neuss'): 75,
    ('Schwerte', 'Schwerte'): 0,
}

inv_capacity = {
    'Bochum': 10,
    'Boenen': 7,
    'Dortmund': 12,
    'Gelsenkirchen': 10,
    'Hagen': 12,
    'Iserlohn': 9,
    'Neuss': 8,
    'Schwerte': 5,
}


# Preprocessing

In [3]:
# -------------------------------------------------------------------------
# Preprocessing: weights, patterns, tightening parameters
# -------------------------------------------------------------------------
radius = diameter['A'] / 2
weights_lb = {l: math.pi * (radius**2) * length_lb[l] * density for l in long_bar_types}
weights_rb = {r: math.pi * (radius**2) * length_rb[r] * density for r in rebar_types}

def generate_patterns(long_length, rebar_lengths):
    patterns = []
    max_A = int(long_length // rebar_lengths['A'])
    max_B = int(long_length // rebar_lengths['B'])
    max_C = int(long_length // rebar_lengths['C'])

    for a in range(max_A + 1):
        for b in range(max_B + 1):
            for c in range(max_C + 1):
                total_len = a * rebar_lengths['A'] + b * rebar_lengths['B'] + c * rebar_lengths['C']
                if 0 < total_len <= long_length:
                    patterns.append({
                        'A': a,
                        'B': b,
                        'C': c,
                        'used_len': total_len
                    })
    return patterns

def filter_nondominated_patterns(patterns):
    nondom = []
    for p in patterns:
        dominated = False
        for q in patterns:
            if p is q:
                continue

            produces_more_or_equal = (
                q['A'] >= p['A'] and
                q['B'] >= p['B'] and
                q['C'] >= p['C']
            )
            uses_less_or_equal_length = q['used_len'] <= p['used_len']
            strictly_better_somewhere = (
                q['A'] > p['A'] or
                q['B'] > p['B'] or
                q['C'] > p['C'] or
                q['used_len'] < p['used_len']
            )

            if produces_more_or_equal and uses_less_or_equal_length and strictly_better_somewhere:
                dominated = True
                break

        if not dominated:
            nondom.append({'A': p['A'], 'B': p['B'], 'C': p['C']})
    return nondom

patterns_1 = filter_nondominated_patterns(generate_patterns(length_lb['1'], length_rb))
patterns_2 = filter_nondominated_patterns(generate_patterns(length_lb['2'], length_rb))

print(f"Number of patterns for 9m bars:  {len(patterns_1)}")
print(f"Number of patterns for 12m bars: {len(patterns_2)}")

# Remaining demand from period t onward
remaining_demand = {}
future_demand = {}

for r in rebar_types:
    for c in customer_areas:
        for t in range(1, numOfPeriods + 1):
            remaining_demand[r, c, t] = sum(demand[r, tau, c] for tau in range(t, numOfPeriods + 1))
            if t < numOfPeriods:
                future_demand[r, c, t] = sum(demand[r, tau, c] for tau in range(t + 1, numOfPeriods + 1))
            else:
                future_demand[r, c, t] = 0

# Tight M values for production activation
M1 = {f: math.floor(steel_capacity[f] / weights_lb['1']) for f in factories}
M2 = {f: math.floor(steel_capacity[f] / weights_lb['2']) for f in factories}

Number of patterns for 9m bars:  12
Number of patterns for 12m bars: 23


# Generate all Routes

In [4]:
def route_distance(factory, seq, dist):
    """
    Distance of closed route:
    factory -> seq[0] -> seq[1] -> ... -> seq[-1] -> factory
    """
    total = dist[(factory, seq[0])]
    for i in range(len(seq) - 1):
        total += dist[(seq[i], seq[i+1])]
    total += dist[(seq[-1], factory)]
    return total

route_data = {}            # route_data[f][rid] = {'seq': ..., 'customers': ..., 'cost': ...}
route_ids_by_factory = {}  # route_ids_by_factory[f] = [rid1, rid2, ...]

for f in factories:
    route_data[f] = {}
    route_ids_by_factory[f] = []
    rid = 0

    # all ordered routes of length 1 to 8
    for m in range(1, len(customer_areas) + 1):
        for seq in permutations(customer_areas, m):
            route_data[f][rid] = {
                'seq': seq,
                'customers': tuple(seq),
                'cost': route_distance(f, seq, dist)
            }
            route_ids_by_factory[f].append(rid)
            rid += 1

    print(f"Factory {f}: generated {rid} routes")

FR = [(f, r) for f in factories for r in route_ids_by_factory[f]]

Factory IJmuiden: generated 109600 routes
Factory Segal: generated 109600 routes
Factory South Wales: generated 109600 routes


# Sets and Parameters

In [5]:
# -------------------------------------------------------------------------
# Model
# -------------------------------------------------------------------------
model = pe.ConcreteModel(name="Production_Delivery_Planning_RouteBased_2Vehicles_AllRoutes")

# Sets
model.F = pe.Set(initialize=factories)
model.C = pe.Set(initialize=customer_areas)
model.T = pe.Set(initialize=range(1, numOfPeriods + 1))
model.K = pe.Set(initialize=vehicles)
model.L = pe.Set(initialize=long_bar_types)
model.RB = pe.Set(initialize=rebar_types)
model.P1 = pe.Set(initialize=range(len(patterns_1)))
model.P2 = pe.Set(initialize=range(len(patterns_2)))
model.FR = pe.Set(dimen=2, initialize=FR)

# Parameters
model.demand = pe.Param(
    model.RB, model.T, model.C,
    initialize=lambda m, rb, t, c: demand.get((rb, t, c), 0)
)
model.steel_capacity = pe.Param(model.F, initialize=steel_capacity)
model.vehicle_capacity = pe.Param(initialize=vehicle_capacity)
model.f_fleet = pe.Param(model.F, initialize=f_fleet, within=pe.NonNegativeReals)
model.c_fleet = pe.Param(model.F, initialize=c_fleet, within=pe.NonNegativeReals)
model.weights_lb = pe.Param(model.L, initialize=weights_lb)
model.weights_rb = pe.Param(model.RB, initialize=weights_rb)
model.inv_capacity = pe.Param(model.C, initialize=inv_capacity)
model.remaining_demand = pe.Param(model.RB, model.C, model.T, initialize=remaining_demand)
model.future_demand = pe.Param(model.RB, model.C, model.T, initialize=future_demand)

def route_cost_init(m, f, r):
    return route_data[f][r]['cost']
model.route_cost = pe.Param(model.FR, initialize=route_cost_init)

# route_visit[f,r,c] = 1 if customer c is on route r of factory f
route_visit_dict = {}
for f in factories:
    for r in route_ids_by_factory[f]:
        customers_in_route = set(route_data[f][r]['customers'])
        for c in customer_areas:
            route_visit_dict[(f, r, c)] = 1 if c in customers_in_route else 0

model.route_visit = pe.Param(model.FR, model.C, initialize=route_visit_dict, within=pe.Binary)

# Decision Variables and Objective Function

In [6]:
# -------------------------------------------------------------------------
# Decision Variables
# -------------------------------------------------------------------------
# Production
model.x1 = pe.Var(model.F, model.P1, model.T, domain=pe.NonNegativeIntegers)
model.x2 = pe.Var(model.F, model.P2, model.T, domain=pe.NonNegativeIntegers)

# Route selection
# lam[f,r,k,t] = 1 if vehicle k of factory f chooses route r in period t
model.lam = pe.Var(model.FR, model.K, model.T, domain=pe.Binary)

# Factory active
# w[f,t] = 1 if factory f is active in period t
model.w = pe.Var(model.F, model.T, domain=pe.Binary)

# Customer inventory
model.I = pe.Var(model.RB, model.C, model.T, domain=pe.NonNegativeIntegers)

# Delivered rebars
# q[f,k,rb,c,t] = number of rebars type rb delivered by vehicle k of factory f to customer c in period t
model.q = pe.Var(model.F, model.K, model.RB, model.C, model.T, domain=pe.NonNegativeIntegers)

# -------------------------------------------------------------------------
# Objective Function
# -------------------------------------------------------------------------

def obj_rule(m):
    routing_cost = sum(
        m.lam[f, r, k, t] * m.route_cost[f, r] * m.c_fleet[f]
        for (f, r) in m.FR for k in m.K for t in m.T
    )

    fixed_cost = sum(
        m.w[f, t] * m.f_fleet[f]
        for f in m.F for t in m.T
    )

    return routing_cost + fixed_cost

model.Obj = pe.Objective(rule=obj_rule, sense=pe.minimize)

# Constraints

In [7]:
# -------------------------------------------------------------------------
# Constraints
# -------------------------------------------------------------------------
# -------------------------------------------------------------------------
# Each vehicle can select at most one route per period
# -------------------------------------------------------------------------
def one_route_per_vehicle_rule(m, f, k, t):
    return sum(m.lam[f, r, k, t] for r in route_ids_by_factory[f]) <= 1
model.OneRoutePerVehicle = pe.Constraint(model.F, model.K, model.T, rule=one_route_per_vehicle_rule)

# -------------------------------------------------------------------------
# If a route is selected, the factory must be active
# -------------------------------------------------------------------------
def route_implies_factory_active_rule(m, f, r, k, t):
    return m.lam[f, r, k, t] <= m.w[f, t]
model.RouteImpliesFactoryActive = pe.Constraint(model.FR, model.K, model.T, rule=route_implies_factory_active_rule)

# -------------------------------------------------------------------------
# A customer can be visited by at most one vehicle in one period
# (across all factories and all vehicles)
# -------------------------------------------------------------------------
def unique_visit_rule(m, c, t):
    return sum(
        m.route_visit[f, r, c] * m.lam[f, r, k, t]
        for f in m.F for k in m.K for r in route_ids_by_factory[f]
    ) <= 1
model.UniqueVisit = pe.Constraint(model.C, model.T, rule=unique_visit_rule)

# -------------------------------------------------------------------------
# Deliver only to customers on the selected route
# -------------------------------------------------------------------------
def ship_only_if_on_selected_route_rule(m, f, k, rb, c, t):
    max_rb_in_truck = math.floor(vehicle_capacity / weights_rb[rb])
    return m.q[f, k, rb, c, t] <= max_rb_in_truck * sum(
        m.route_visit[f, r, c] * m.lam[f, r, k, t]
        for r in route_ids_by_factory[f]
    )
model.ShipOnlyIfOnSelectedRoute = pe.Constraint(
    model.F, model.K, model.RB, model.C, model.T,
    rule=ship_only_if_on_selected_route_rule
)

# -------------------------------------------------------------------------
# If a customer is visited on the selected route, at least 1 rebar must be delivered
# -------------------------------------------------------------------------
def deliver_to_all_visited_rule(m, f, k, c, t):
    return sum(m.q[f, k, rb, c, t] for rb in m.RB) >= sum(
        m.route_visit[f, r, c] * m.lam[f, r, k, t]
        for r in route_ids_by_factory[f]
    )
model.DeliverToAllVisited = pe.Constraint(
    model.F, model.K, model.C, model.T,
    rule=deliver_to_all_visited_rule
)

# -------------------------------------------------------------------------
# Vehicle capacity per vehicle
# -------------------------------------------------------------------------
def vehicle_cap_rule(m, f, k, t):
    shipped_weight = sum(
        m.q[f, k, rb, c, t] * m.weights_rb[rb]
        for rb in m.RB for c in m.C
    )
    return shipped_weight <= m.vehicle_capacity
model.VehicleCapConstraint = pe.Constraint(model.F, model.K, model.T, rule=vehicle_cap_rule)

# -------------------------------------------------------------------------
# Production must cover total shipments of each rebar type
# -------------------------------------------------------------------------
def link_prod_rule(m, f, rb, t):
    prod1 = sum(patterns_1[p][rb] * m.x1[f, p, t] for p in m.P1)
    prod2 = sum(patterns_2[p][rb] * m.x2[f, p, t] for p in m.P2)
    shipped = sum(m.q[f, k, rb, c, t] for k in m.K for c in m.C)
    return prod1 + prod2 >= shipped
model.LinkProdConstraint = pe.Constraint(model.F, model.RB, model.T, rule=link_prod_rule)

# -------------------------------------------------------------------------
# Inventory balance at customer
# -------------------------------------------------------------------------
def inv_balance_rule(m, rb, c, t):
    inbound = sum(m.q[f, k, rb, c, t] for f in m.F for k in m.K)

    if t == 1:
        return m.I[rb, c, t] == inbound - m.demand[rb, t, c]
    else:
        return m.I[rb, c, t] == m.I[rb, c, t-1] + inbound - m.demand[rb, t, c]
model.InvBalance = pe.Constraint(model.RB, model.C, model.T, rule=inv_balance_rule)

# -------------------------------------------------------------------------
# Inventory capacity at customer
# -------------------------------------------------------------------------
def inv_cap_rule(m, c, t):
    return sum(m.I[rb, c, t] * m.weights_rb[rb] for rb in m.RB) <= m.inv_capacity[c]
model.InvCap = pe.Constraint(model.C, model.T, rule=inv_cap_rule)

# -------------------------------------------------------------------------
# Factory steel capacity if active
# -------------------------------------------------------------------------
def factory_cap_if_active_rule(m, f, t):
    weight_1 = sum(m.x1[f, p, t] * m.weights_lb['1'] for p in m.P1)
    weight_2 = sum(m.x2[f, p, t] * m.weights_lb['2'] for p in m.P2)
    return weight_1 + weight_2 <= m.steel_capacity[f] * m.w[f, t]
model.FactoryCapIfActive = pe.Constraint(model.F, model.T, rule=factory_cap_if_active_rule)

# -------------------------------------------------------------------------
# Optional tightening:
# if factory is inactive, no routes;
# if routes are used, w must become 1
# -------------------------------------------------------------------------
def active_if_any_route_rule(m, f, t):
    return sum(m.lam[f, r, k, t] for r in route_ids_by_factory[f] for k in m.K) <= len(vehicles) * m.w[f, t]
model.ActiveIfAnyRoute = pe.Constraint(model.F, model.T, rule=active_if_any_route_rule)

# Tight production activation constraints
def prod_only_if_active_1_rule(m, f, t):
    return sum(m.x1[f, p, t] for p in m.P1) <= M1[f] * m.w[f, t]
model.ProdOnlyIfActive1 = pe.Constraint(model.F, model.T, rule=prod_only_if_active_1_rule)

def prod_only_if_active_2_rule(m, f, t):
    return sum(m.x2[f, p, t] for p in m.P2) <= M2[f] * m.w[f, t]
model.ProdOnlyIfActive2 = pe.Constraint(model.F, model.T, rule=prod_only_if_active_2_rule)

def final_inventory_rule(m, rb, c):
    return m.I[rb, c, numOfPeriods] == 0
model.FinalInventory = pe.Constraint(model.RB, model.C, rule=final_inventory_rule)

def inv_future_demand_rule(m, rb, c, t):
    return m.I[rb, c, t] <= m.future_demand[rb, c, t]


# Solve & Print Results

In [8]:
# -------------------------------------------------------------------------
# Solve
# -------------------------------------------------------------------------
solver = po.SolverFactory('gurobi')
result = solver.solve(model, tee=True, options={'TimeLimit': 7200, 'MIPGap': 0.03})

print(f"\nOptimization Status: {result.solver.status}")
print(f"Termination Condition: {result.solver.termination_condition}")

if (result.solver.status == po.SolverStatus.ok or
    result.solver.termination_condition in [po.TerminationCondition.optimal,
                                            po.TerminationCondition.maxTimeLimit,
                                            po.TerminationCondition.feasible]):
    print(f"Total Objective Cost: €{pe.value(model.Obj):.2f}")

# -------------------------------------------------------------------------
# Print results
# -------------------------------------------------------------------------
print("\nOptimal production, delivery and inventory plan (route-based, 2 vehicles/factory)")

for t in model.T:
    print(f"\n================ PERIOD {t} ================")

    for f in model.F:
        if pe.value(model.w[f, t]) > 0.5:
            print(f"\nFactory: {f}")

            # ---------------------------
            # Production
            # ---------------------------
            total_steel_used = 0
            print("  Production:")

            for p in model.P1:
                val = pe.value(model.x1[f, p, t])
                if val is not None and val > 1e-6:
                    bars = int(round(val))
                    total_steel_used += bars * weights_lb['1']
                    print(f"    • {bars}x 9m bars with pattern {patterns_1[p]}")

            for p in model.P2:
                val = pe.value(model.x2[f, p, t])
                if val is not None and val > 1e-6:
                    bars = int(round(val))
                    total_steel_used += bars * weights_lb['2']
                    print(f"    • {bars}x 12m bars with pattern {patterns_2[p]}")

            print(f"    Steel used: {total_steel_used:.2f} / {steel_capacity[f]} tonnes")

            # ---------------------------
            # Vehicle routes
            # ---------------------------
            for k in model.K:
                chosen_routes = [r for r in route_ids_by_factory[f] if pe.value(model.lam[f, r, k, t]) > 0.5]

                if chosen_routes:
                    r = chosen_routes[0]
                    seq = route_data[f][r]['seq']
                    route_str = " -> ".join([f] + list(seq) + [f])

                    print(f"\n  Vehicle {k}:")
                    print(f"    Route: {route_str}")
                    print(f"    Route distance: {route_data[f][r]['cost']} km")

                    total_load = 0
                    print("    Shipments:")

                    for c in model.C:
                        qtyA = int(round(pe.value(model.q[f, k, 'A', c, t])))
                        qtyB = int(round(pe.value(model.q[f, k, 'B', c, t])))
                        qtyC = int(round(pe.value(model.q[f, k, 'C', c, t])))

                        if qtyA + qtyB + qtyC > 0:
                            print(f"      {c}: A={qtyA}, B={qtyB}, C={qtyC}")
                            total_load += (
                                qtyA * weights_rb['A'] +
                                qtyB * weights_rb['B'] +
                                qtyC * weights_rb['C']
                            )

                    print(f"    Vehicle load: {total_load:.2f} / {vehicle_capacity} tonnes")

    # ---------------------------
    # Inventory per customer
    # ---------------------------
    print("\nEnd Inventory (rebars):")

    for c in model.C:
        invA = int(round(pe.value(model.I['A', c, t])))
        invB = int(round(pe.value(model.I['B', c, t])))
        invC = int(round(pe.value(model.I['C', c, t])))

        if invA + invB + invC > 0:
            total_inv_weight = (
                invA * weights_rb['A'] +
                invB * weights_rb['B'] +
                invC * weights_rb['C']
            )

            print(f"  {c}: A={invA}, B={invB}, C={invC} "
                  f"({total_inv_weight:.2f} / {inv_capacity[c]} tonnes)")

print(f"\nTotal Objective Cost: €{pe.value(model.Obj):.2f}")

Read LP format model from file C:\Users\Eveli\AppData\Local\Temp\tmpmy067aq8.pyomo.lp
Reading time = 64.32 seconds
x1: 2631484 rows, 2631504 columns, 102591264 nonzeros
Set parameter TimeLimit to value 7200
Set parameter MIPGap to value 0.03
Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 7 5700U with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 2631484 rows, 2631504 columns and 102591264 nonzeros
Model fingerprint: 0x13c3699a
Variable types: 0 continuous, 2631504 integer (2630412 binary)
Coefficient statistics:
  Matrix range     [5e-02, 2e+02]
  Objective range  [1e+02, 7e+03]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 4e+01]
Presolve removed 0 rows and 0 columns (presolve time = 5s) ...
Presolve removed 24 rows and 24 columns (presolve time = 11s) ...
Presolve removed 46 rows and 24 columns (presolve time 

H 3590  1435                    18485.000000 14955.3036  19.1%   129  305s
  3876  1730 15691.5221  109  331 18485.0000 14955.3036  19.1%   128  311s
  4149  2059 15707.9568  123  352 18485.0000 14955.3036  19.1%   131  315s
  4611  2320 16024.1897  137  264 18485.0000 14955.3036  19.1%   128  320s
H 4819  2498                    18446.000000 14955.3036  18.9%   128  323s
H 4846  2491                    18422.000000 14955.3036  18.8%   129  324s
  5013  2836 15735.8152  154  337 18422.0000 14955.3036  18.8%   128  327s
  5387  2944 15737.8245  162  341 18422.0000 14955.3036  18.8%   125  331s
  5868  3388 15763.4740  172  367 18422.0000 14955.3036  18.8%   125  338s
H 5888  3388                    18416.000000 14955.3036  18.8%   126  338s
H 5914  3372                    18359.000000 14955.3036  18.5%   126  339s
H 5957  3370                    18347.000000 14955.3036  18.5%   126  339s
  6037  3552 15822.7093  178  385 18347.0000 14955.3036  18.5%   125  341s
H 6098  3498             

 31349 21281 16504.0687   45  297 18182.0000 15933.8707  12.4%  94.8  875s
 31499 21324 16766.7421   52  290 18182.0000 15933.8707  12.4%  95.4  880s
 31653 21384 16794.1514   62  243 18182.0000 15933.8707  12.4%  96.3  886s
H31676 20316                    18155.000000 15933.8707  12.2%  96.6  886s
H31751 19377                    18152.000000 15933.8707  12.2%  96.7  889s
H31787 18407                    18149.000000 15933.8707  12.2%  97.0  889s
 31802 18455 16813.5657   71  262 18149.0000 15933.8707  12.2%  97.0  890s
H31887 17567                    18146.000000 15933.8707  12.2%  97.4  892s
 32001 17648 16823.8519   80  266 18146.0000 15933.8707  12.2%  97.8  895s
 32220 17782 17002.9563   91  182 18146.0000 15933.8707  12.2%  98.3  900s
 32507 17964 16871.2272  104  252 18146.0000 15933.8707  12.2%  99.0  906s
 32638 18004 16895.6426  111  294 18146.0000 15933.8707  12.2%  99.3  910s
 33005 18235 16903.7580  130  281 18146.0000 15933.8707  12.2%   100  916s
 33324 18404 16910.6174  

 116162 71940 17032.7504  317  306 18146.0000 16241.8602  10.5%  93.9 1501s
 117027 72459 17037.1550  332  294 18146.0000 16241.8602  10.5%  94.0 1506s
 117668 73180 infeasible  344      18146.0000 16241.8602  10.5%  94.0 1511s
 118407 74296 infeasible  364      18146.0000 16241.8602  10.5%  94.0 1517s
 119571 74969 17074.2154  384  304 18146.0000 16241.8602  10.5%  93.6 1522s
 120377 75566 17093.9209  404  301 18146.0000 16241.8602  10.5%  93.6 1530s
 121078 76790 17106.9374  426  287 18146.0000 16241.8602  10.5%  93.7 1538s
 122515 77491 17119.1556  469  289 18146.0000 16241.8602  10.5%  93.5 1542s
 123355 78093 17219.4472  492  214 18146.0000 16241.8602  10.5%  93.4 1547s
 124091 78850 17441.9533  511  257 18146.0000 16241.8602  10.5%  93.4 1552s
 124972 79525 17156.5650  531  268 18146.0000 16241.8602  10.5%  93.3 1557s
 125698 80307 17162.8846  546  286 18146.0000 16241.8602  10.5%  93.3 1602s
 126542 80834 17368.7323  566  222 18146.0000 16241.8602  10.5%  93.2 1608s
 127208 8176

 216437 160385 17544.8461   59  243 18146.0000 16358.4316  9.85%  82.7 2121s
 217187 160772 16550.0387   45  353 18146.0000 16358.4316  9.85%  82.7 2126s
 217798 161182 16598.8940   54  320 18146.0000 16358.4316  9.85%  82.7 2130s
 218378 161774 17124.6338   70  288 18146.0000 16358.4316  9.85%  82.8 2135s
 220048 162742 17225.6168  197  281 18146.0000 16358.4316  9.85%  82.8 2144s
 220670 163295 17246.0845  219  240 18146.0000 16358.4316  9.85%  83.0 2148s
 221331 163685 17997.0999  242  193 18146.0000 16358.4316  9.85%  83.1 2153s
 221823 164332 17282.4396  261  235 18146.0000 16358.4316  9.85%  83.1 2157s
 222585 164758 17302.5511  298  239 18146.0000 16358.4316  9.85%  83.2 2162s
 223185 165594 17386.7271  332  153 18146.0000 16358.4316  9.85%  83.3 2167s
 224122 166334 17359.2306  366  245 18146.0000 16358.4316  9.85%  83.3 2174s
 225005 167018 17395.0596  401  235 18146.0000 16358.4316  9.85%  83.2 2180s
 225903 167496 18131.9227  443  179 18146.0000 16358.4316  9.85%  83.2 2185s

...

 400815 316510 17835.5310   51  246 18134.0000 16413.8445  9.49%  83.7 3389s
 401813 317331 17923.3053   47  287 18134.0000 16413.9691  9.49%  83.6 3396s
 402831 317822 16548.2385  107  190 18134.0000 16413.9691  9.49%  83.6 3401s
 403399 318346 17472.7861  164  215 18134.0000 16413.9691  9.49%  83.7 3407s
 403995 318646 17458.8352  255  231 18134.0000 16413.9691  9.49%  83.7 3416s
 404334 319416 17597.3533  320  197 18134.0000 16414.9187  9.48%  83.7 3423s
 405480 319929 17145.3692  147  230 18134.0000 16415.7798  9.48%  83.7 3430s
 406401 320524 17246.2304   45  320 18134.0000 16415.7798  9.48%  83.7 3437s
 407316 320911 17556.3221  143  320 18134.0000 16415.7798  9.48%  83.8 3442s
 407864 321502 17694.4837  204  281 18134.0000 16415.7798  9.48%  83.8 3448s
 408508 322480 17729.9056  268  303 18134.0000 16415.7798  9.48%  83.8 3455s
 409621 323269 17804.6468  379  291 18134.0000 16415.7798  9.48%  83.8 3462s
 410500 323978 17928.2574  485  306 18134.0000 16417.2748  9.47%  83.8 3468s

 490343 391158 18051.1323   61  263 18134.0000 16429.7901  9.40%  84.4 4146s
 491269 391647 16849.2124   59  186 18134.0000 16429.8282  9.40%  84.4 4152s
 491849 392456 16604.6021   89  281 18134.0000 16430.0972  9.40%  84.4 4160s
 492726 393139 17052.7148   72  256 18134.0000 16430.0972  9.40%  84.4 4167s
 493499 393788 16916.4212  148  266 18134.0000 16430.0972  9.40%  84.5 4176s
 494229 394506 17042.2526  200  253 18134.0000 16430.0972  9.40%  84.5 4183s
 495185 395125 17101.2601  295  234 18134.0000 16430.0972  9.40%  84.5 4191s
 495972 396153 17169.2062  384  254 18134.0000 16430.0972  9.40%  84.6 4201s
 497342 396863 17290.4499  491  220 18134.0000 16430.1111  9.40%  84.6 4208s
 498314 397449 16679.6793   38  404 18134.0000 16430.7018  9.39%  84.6 4216s
 499121 397795 18076.0637  120  267 18134.0000 16430.7826  9.39%  84.6 4222s
 499674 398176 17220.7160  186  273 18134.0000 16431.1454  9.39%  84.7 4228s
 500138 398739 17646.3008  112  308 18134.0000 16431.3181  9.39%  84.7 4235s

 576268 462361 18076.4802  147  246 18134.0000 16440.5384  9.34%  85.3 4962s
 577022 462890 16934.6963   51  329 18134.0000 16440.6877  9.34%  85.4 4969s
 577658 463276 18043.5042   53  196 18134.0000 16440.9396  9.34%  85.4 4975s
 578144 464006 17960.1292   77  273 18134.0000 16441.0295  9.34%  85.5 4982s
 579013 464598 17772.9393   93  262 18134.0000 16441.1338  9.34%  85.4 4989s
 579696 465718 17406.0753   98  175 18134.0000 16441.2455  9.33%  85.5 5001s
 580998 466007 infeasible   81      18134.0000 16441.4190  9.33%  85.4 5007s
 581405 466819 18129.9059   68  244 18134.0000 16441.6337  9.33%  85.5 5014s
 582332 467262 16856.7912   73  320 18134.0000 16441.8028  9.33%  85.5 5021s
 582862 467773 16903.9192   92  249 18134.0000 16441.8028  9.33%  85.5 5028s
 583516 468227 17181.5363  127  313 18134.0000 16441.8028  9.33%  85.6 5035s
 584142 468799 17319.7927  151  250 18134.0000 16441.8028  9.33%  85.6 5044s
 584961 469615 17327.6978  173  263 18134.0000 16441.8028  9.33%  85.7 5054s

 669679 535345 16470.4135   53  364 18134.0000 16451.8798  9.28%  86.7 5930s
 670247 536122 17388.8971   95  277 18134.0000 16451.9151  9.28%  86.8 5945s
 671667 536635     cutoff   76      18134.0000 16452.0574  9.28%  86.8 5958s
 673207 537233 17945.9049   66  221 18134.0000 16452.0691  9.28%  86.8 5967s
 674194 537724 16548.6298  138  325 18134.0000 16452.2351  9.27%  86.8 5975s
 674870 538621 16499.0816   58  309 18134.0000 16452.3421  9.27%  86.8 5987s
 676021 539412 16717.8546   50  330 18134.0000 16452.3421  9.27%  86.8 5996s
 676985 540129 17553.7490  139  232 18134.0000 16452.5010  9.27%  86.8 6005s
 677879 540576 16915.3549   55  229 18134.0000 16452.5010  9.27%  86.8 6013s
 678471 541486 17347.6515   71  328 18134.0000 16452.5010  9.27%  86.8 6023s
 679505 542357 17542.6753  197  276 18134.0000 16452.5234  9.27%  86.8 6034s
 680789 543356 infeasible  142      18134.0000 16452.6022  9.27%  86.8 6045s
 681998 544061 17838.6850   88  316 18134.0000 16452.7516  9.27%  86.8 6054s

 764313 608827 17031.3671  140  288 18134.0000 16461.0301  9.23%  87.5 7044s
 764764 609500 16915.5280  142  325 18134.0000 16461.0967  9.23%  87.5 7056s
 765737 609921 infeasible   45      18134.0000 16461.1811  9.22%  87.5 7068s
 766713 610493 16854.7462  103  222 18134.0000 16461.2913  9.22%  87.6 7081s
 767517 611075 17524.3170  109  293 18134.0000 16461.3935  9.22%  87.6 7092s
 768342 611907 16981.0456   54  285 18134.0000 16461.3935  9.22%  87.6 7105s
 769260 612891 17117.6888  170  334 18134.0000 16461.3935  9.22%  87.6 7118s
 770367 613378 17186.6728  291  345 18134.0000 16461.3935  9.22%  87.6 7128s
 770945 613910 17333.1525  374  278 18134.0000 16461.3935  9.22%  87.6 7140s
 771745 614566 17872.9293  463  329 18134.0000 16461.3935  9.22%  87.7 7151s
 772549 615064 17987.7853  585  325 18134.0000 16461.4126  9.22%  87.6 7162s
 773235 615645 16863.7357   91  285 18134.0000 16461.4126  9.22%  87.7 7173s
 774063 616530 16962.8418  176  272 18134.0000 16461.9522  9.22%  87.7 7189s